<a href="https://colab.research.google.com/github/nithinrb/NVIDIA/blob/main/DAY_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain malware analysis in 100 words."
)

print(response.text)

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

text = input("Enter text: ")

try:
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    ) as response:
        response.stream_to_file("output.mp3")

    print("Audio saved as output.mp3")

except Exception as e:
    print("Error:")
    print(e)

In [ ]:
from google import genai
from google.genai import types
from pydub import AudioSegment
import os
from io import BytesIO
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.genai.errors import ServerError # Import ServerError
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY') # Using Colab's Secrets Manager
)

question = input("Ask Gemini: ")

# Define a retry function for generate_content
@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(ServerError) # Retry only on ServerError (like 503)
)
def generate_content_with_retry(model, contents):
    return client.models.generate_content(
        model=model,
        contents=contents
    )

# Step 1: Generate answer with retry logic
try:
    response = generate_content_with_retry(
        model="gemini-2.5-flash",
        contents=question
    )
    answer = response.text

    print("\nGemini Response:\n")
    print(answer)

    # Step 2: Generate speech with retry logic
    @retry(
        wait=wait_exponential(multiplier=1, min=4, max=10),
        stop=stop_after_attempt(5),
        retry=retry_if_exception_type(ServerError) # Retry only on ServerError
    )
    def generate_tts_with_retry(model, contents, config):
        return client.models.generate_content(
            model=model,
            contents=contents,
            config=config
        )

    tts_response = generate_tts_with_retry(
        model="gemini-2.5-flash-preview-tts",
        contents=[types.Part(text=answer)], # Wrap the text in types.Part for TTS
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"]
        )
    )

    audio_bytes = (
        tts_response.candidates[0]
        .content.parts[0]
        .inline_data.data
    )

    # Load raw audio data using from_raw with assumed parameters (LINEAR16: 24kHz, 16-bit, mono)
    audio = AudioSegment.from_raw(
        BytesIO(audio_bytes),
        sample_width=2,  # 16-bit audio = 2 bytes per sample
        frame_rate=24000, # Common sample rate for TTS LINEAR16
        channels=1,      # Mono audio
        format='s16le'   # Signed 16-bit Little-Endian PCM
    )

    # Convert to MP3
    audio.export("gemini_response.mp3", format="mp3")

    print("\nSaved as gemini_response.mp3")

except ServerError as e:
    print(f"\nFailed to generate content after multiple retries due to: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

In [ ]:
from google import genai
from google.genai import types
from pydub import AudioSegment
import os
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY')
)

question = input("Ask Gemini: ")

# Step 1: Generate answer
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question
)

answer = response.text

print("\nGemini Response:\n")
print(answer)

# Clean the answer text for TTS
# Remove markdown bolding (**) and bullet points (*   )
cleaned_answer = answer.replace('**', '').replace('*   ', '')
# Replace multiple newlines with a single space to make it flow as continuous speech
cleaned_answer = ' '.join(cleaned_answer.splitlines())
cleaned_answer = cleaned_answer.strip()

# Step 2: Generate speech
tts_response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=[types.Part(text=cleaned_answer)], # Wrap the cleaned text in types.Part for TTS
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"]
    )
)

audio_bytes = (
    tts_response.candidates[0]
    .content.parts[0]
    .inline_data.data
)

# Save temporary WAV
with open("temp.wav", "wb") as f:
    f.write(audio_bytes)

# Convert WAV → MP3
audio = AudioSegment.from_wav("temp.wav")
audio.export("gemini_response.mp3", format="mp3")

print("\nSaved as gemini_response.mp3")

In [ ]:
import time
import psutil
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_TEXT = """
Machine learning techniques are increasingly used for phishing detection.
Transformers provide strong contextual understanding.
"""

# -------------------------
# Load Models
# -------------------------

bert_model_name = "bert-base-uncased"
gpt_model_name = "gpt2"

print("Loading Models...")

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(
    bert_model_name
).to(DEVICE)

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)

if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_model = AutoModelForCausalLM.from_pretrained(
    gpt_model_name
).to(DEVICE)

# -------------------------
# Benchmark Function
# -------------------------

def benchmark(model, tokenizer, text, model_type):

    process = psutil.Process()

    memory_before = process.memory_info().rss / 1024**2

    tokens = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    tokens = {k:v.to(DEVICE) for k,v in tokens.items()}

    start = time.time()

    with torch.no_grad():

        if model_type == "BERT":
            output = model(**tokens)

        else:
            output = model.generate(
                tokens["input_ids"],
                max_new_tokens=50
            )

    end = time.time()

    memory_after = process.memory_info().rss / 1024**2

    total_tokens = tokens["input_ids"].shape[1]

    return {
        "Latency(sec)": round(end-start,4),
        "Tokens": total_tokens,
        "Memory(MB)": round(
            memory_after-memory_before,
            2
        ),
        "Tokens/sec":
            round(total_tokens/(end-start),2)
    }

# -------------------------
# Run Benchmarks
# -------------------------

bert_results = benchmark(
    bert_model,
    bert_tokenizer,
    TEST_TEXT,
    "BERT"
)

gpt_results = benchmark(
    gpt_model,
    gpt_tokenizer,
    TEST_TEXT,
    "GPT"
)

# -------------------------
# Print Results
# -------------------------

print("\n===== Benchmark Results =====")

print("\nBERT")
for k,v in bert_results.items():
    print(k,":",v)

print("\nGPT")
for k,v in gpt_results.items():
    print(k,":",v)

print("\nModel Info")

print("BERT Context Window: 512")
print("GPT2 Context Window: 1024")

print("Device:", DEVICE)

In [ ]:
import os
import time
import psutil
import pandas as pd

from openai import OpenAI
from google import genai
from google.colab import userdata

# -----------------------------
# API Clients
# -----------------------------

openai_client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

gemini_client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY')
)

PROMPTS = [
    "Explain phishing attacks",
    "Write python code for bubble sort",
    "Explain transformers in AI"
]

# -----------------------------
# Benchmark Function
# -----------------------------

def benchmark_gpt3(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = openai_client.completions.create(
        model="davinci-002",
        prompt=prompt,
        max_tokens=150
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.choices[0].text

    return {
        "model":"GPT-3",
        "latency_sec":round(end-start,3),
        "memory_mb":round(
            mem_after-mem_before,
            2
        ),
        "output_chars":len(output),
        "response":output[:150]
    }


def benchmark_gemini(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.text

    return {
        "model":"Gemini-2",
        "latency_sec":round(end-start,3),
        "memory_mb":round(
            mem_after-mem_before,
            2
        ),
        "output_chars":len(output),
        "response":output[:150]
    }

# -----------------------------
# Execute Tests
# -----------------------------

results = []

for p in PROMPTS:

    print("\nTesting:", p)

    results.append(
        benchmark_gpt3(p)
    )

    results.append(
        benchmark_gemini(p)
    )

df = pd.DataFrame(results)

print("\n========== RESULTS ==========\n")

print(df)

df.to_csv(
    "benchmark_results.csv",
    index=False
)

print(
    "\nSaved benchmark_results.csv"
)

In [ ]:
import os
import time
import psutil
import pandas as pd

from openai import OpenAI
from google import genai
from google.colab import userdata

# --------------------------------
# Clients
# --------------------------------

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=userdata.get('NVIDIA_API_KEY')
)

gemini_client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY')
)

PROMPTS = [
    "Explain phishing attacks",
    "Write python code for bubble sort",
    "Explain transformers in AI"
]

# --------------------------------
# GPT OSS 20B Benchmark
# --------------------------------

def benchmark_gpt_oss(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = nvidia_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7,
        top_p=1.0,
        max_tokens=512,
        stream=False
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.choices[0].message.content

    return {
        "model": "GPT-OSS-20B",
        "latency_sec": round(end - start, 3),
        "memory_mb": round(mem_after - mem_before, 2),
        "output_chars": len(output),
        "response": output[:150]
    }

# --------------------------------
# Gemini Benchmark
# --------------------------------

def benchmark_gemini(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.text

    return {
        "model": "Gemini-2.5-Flash",
        "latency_sec": round(end - start, 3),
        "memory_mb": round(mem_after - mem_before, 2),
        "output_chars": len(output),
        "response": output[:150]
    }

# --------------------------------
# Run Benchmark
# --------------------------------

results = []

for prompt in PROMPTS:

    print(f"\nTesting: {prompt}")

    results.append(
        benchmark_gpt_oss(prompt)
    )

    results.append(
        benchmark_gemini(prompt)
    )

df = pd.DataFrame(results)

print("\n========== RESULTS ==========\n")
print(df)

df.to_csv(
    "gptoss_vs_gemini_benchmark.csv",
    index=False
)

print(
    "\nSaved gptoss_vs_gemini_benchmark.csv"
)